# Image pattern mining on a local Julia kernel

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NgTHung/Itemset/blob/main/notebooks/image_patterns_colab.ipynb)

Handwritten digits become transactions, and Eclat finds the sets of visual words that appear together in many images. These cells encode the UCI Semeion digits with a seeded patch codebook, mine three thresholds with the team's Eclat, show the images that contain a pattern next to images that lack it, and build the pattern features. Open the notebook from a checkout of the repository in an IDE with a Julia kernel; the first two cells find the checkout and install the project.

The same steps run on Google Colab in `image_patterns_colab.ipynb`, which differs only in setup. `applications/image_patterns/README.md` explains the encoding and the outputs.

In [ ]:
# The notebook sits in notebooks/, so the checkout is its parent unless the working directory already is the checkout.
ROOT = abspath(get(ENV, "ITEMSET_ROOT", isfile("Project.toml") ? pwd() : dirname(pwd())))
isfile(joinpath(ROOT, "Project.toml")) ||
    error("Open the notebook from the repository root or from notebooks/, or set ITEMSET_ROOT to the checkout")
cd(ROOT)
println("Repository: ", ROOT)
println("Revision: ", ispath(".git") ? readchomp(`git rev-parse HEAD`) : "no .git folder; record the commit of this copy")


In [ ]:
using Pkg
v"1.12" <= VERSION < v"2" ||
    error("Use Julia 1.12 or later within 1.x; 1.13.2 gives the recorded results")
Pkg.activate(ROOT)
Pkg.instantiate()

using ItemsetMining
using Random
println("Julia: ", VERSION, " | threads: ", Threads.nthreads(), " | seed: ", DEFAULT_SEED)
println("Project: ", Base.active_project())
artifact_manifest = get(ENV, "ITEMSET_ARTIFACT_MANIFEST", "")
notebook_artifacts = Dict{String,String}()


In [ ]:
semeion_path = joinpath(ROOT, "data", "raw", "semeion", "semeion.data")
pilot_output = joinpath(mktempdir(; cleanup=isempty(artifact_manifest)), "semeion-pilot")
module ImagePilotRun
    include(joinpath(Main.ROOT, "scripts", "image_pilot.jl"))
end
pilot = ImagePilotRun.run_mining_command([semeion_path, pilot_output])
notebook_artifacts["encoding"] = pilot_output
notebook_artifacts["semeion"] = pilot_output
encoding_manifest = ImagePilotRun.SemeionPilot.TOML.parsefile(joinpath(pilot_output, "manifest.toml"))
display(encoding_manifest["statistics"])
display(HTML(read(joinpath(pilot_output, "digits.svg"), String)))
display(HTML(read(joinpath(pilot_output, "visual_words.svg"), String)))


In [ ]:
println(read(joinpath(pilot_output, "selected_patterns.tsv"), String))
display(HTML(read(joinpath(pilot_output, "supporting-0.2.svg"), String)))
display(HTML(read(joinpath(pilot_output, "supporting-0.05.svg"), String)))


### Analyze patterns and construct features

Use one encoding for both the patterns and their image IDs. The table keeps each digit's full denominator, including empty transactions. Counts describe this fitted dataset. A pattern query retrieves images containing all its words; it does not retain patch positions.

For a fixed pattern list `P`, feature `z[i,j]` is 1 exactly when `P[j]` is a subset of image transaction `T[i]`. The hand-worked example below yields `[1,1,0]` and an empty image yields zeros. For a predictive evaluation, split images first, fit the codebook and mine patterns only on training images, and freeze feature column order for test images. The all-data descriptive run below does not measure held-out classification accuracy.


In [ ]:
analysis_encoding = pilot.encoding
analysis_runs = pilot.runs
analysis_patterns = ImagePilotRun.SemeionPilot.select_patterns(last(analysis_runs).patterns)
for pattern in analysis_patterns
    rows = ImagePilotRun.SemeionPilot.pattern_label_summary(analysis_encoding.images, analysis_encoding.transactions, pattern)
    @assert sum(row.support for row in rows) == last(analysis_runs).patterns[pattern]
    println("Pattern ", pattern, " | label | total | support | fraction | source IDs")
    display(rows)
end
feature_columns = sort!(collect(keys(last(analysis_runs).patterns)))
image_features = ImagePilotRun.SemeionPilot.itemset_features(analysis_encoding.transactions, feature_columns)
@assert vec(sum(image_features; dims=1)) == [last(analysis_runs).patterns[p] for p in feature_columns]
println("Image-feature matrix: ", size(image_features), " | fixed columns: ", feature_columns)
example_features = ImagePilotRun.SemeionPilot.itemset_features([[8, 13], Int[]], [(13,), (8, 13), (4, 8, 13)])
@assert example_features == Bool[1 1 0; 0 0 0]
display(example_features)
